# Sid — Data Preprocessing

Merges raw ACS Census tables with MIT Election Lab results into one county-level dataset.

**Input:** Raw ACS 2020 CSVs + MIT countypres 2000–2024  
**Target year:** 2020 presidential election  
**Output:** `../data/processed/merged_county_dataset.csv`



In [1]:
import pandas as pd
import numpy as np
import os

### (2) File paths

In [2]:
MIT_PATH        = "../data/Raw/countypres_2000-2024.csv"

AGE_PATH        = "../data/Raw/demographic/ACSDT5Y2020.B01001-Data.csv"
RACE_PATH       = "../data/Raw/demographic/ACSDT5Y2020.B02001-Data.csv"
EDUCATION_PATH  = "../data/Raw/demographic/ACSDT5Y2020.B15003-Data.csv"

INCOME_PATH     = "../data/Raw/socioeconomic/ACSDT5Y2020.B19013-Data.csv"
POVERTY_PATH    = "../data/Raw/socioeconomic/ACSDT5Y2020.B17001-Data.csv"
EMPLOYMENT_PATH = "../data/Raw/socioeconomic/ACSDT5Y2020.B23025-Data.csv"
HOUSING_PATH    = "../data/Raw/socioeconomic/ACSDT5Y2020.B25001-Data.csv"

OUTPUT_PATH = "../data/processed/merged_county_dataset.csv"

### (3) Load all raw datasets

In [3]:
mit_df        = pd.read_csv(MIT_PATH)

age_df        = pd.read_csv(AGE_PATH)
race_df       = pd.read_csv(RACE_PATH)
education_df  = pd.read_csv(EDUCATION_PATH)

income_df     = pd.read_csv(INCOME_PATH)
poverty_df    = pd.read_csv(POVERTY_PATH)
employment_df = pd.read_csv(EMPLOYMENT_PATH)
housing_df    = pd.read_csv(HOUSING_PATH)

for name, df in [("MIT", mit_df), ("Age", age_df), ("Race", race_df),
                  ("Education", education_df), ("Income", income_df),
                  ("Poverty", poverty_df), ("Employment", employment_df),
                  ("Housing", housing_df)]:
    print(f"{name}: {df.shape}")

MIT: (94151, 12)
Age: (3222, 101)
Race: (3222, 23)
Education: (3222, 53)
Income: (3222, 5)
Poverty: (3222, 121)
Employment: (3222, 17)
Housing: (3222, 5)


### (4) ACS cleaning function

In [4]:
ACS_SENTINEL = -666666666

def clean_acs(df, label):
    # Row 0 is a description row in ACS exports, drop it
    df = df.iloc[1:].copy()

    # Extract 5-digit FIPS from the last 5 characters of GEO_ID
    df["county_fips"] = df["GEO_ID"].astype(str).str[-5:]

    # Keep only estimate columns (ending in E) but exclude GEO_ID, NAME, county_fips
    skip = {"GEO_ID", "NAME", "county_fips"}
    est_cols = list(dict.fromkeys([
        c for c in df.columns
        if str(c).endswith("E") and c not in skip
    ]))

    df = df[["GEO_ID", "NAME", "county_fips"] + est_cols].copy()

    # Convert estimates to numeric, non-numeric becomes NaN
    for col in est_cols:
        if isinstance(df[col], pd.Series):
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Replace ACS suppressed-data sentinel with NaN
    df[est_cols] = df[est_cols].replace(ACS_SENTINEL, np.nan)

    # Zero-pad FIPS to 5 digits
    df["county_fips"] = df["county_fips"].astype(str).str.zfill(5)

    print(f"{label}: {df.shape} | FIPS dupes: {df['county_fips'].duplicated().sum()}")
    return df

### (5) Clean all ACS tables

In [5]:
print("Cleaning ACS tables:")
age_clean        = clean_acs(age_df,        "Age")
race_clean       = clean_acs(race_df,       "Race")
education_clean  = clean_acs(education_df,  "Education")
income_clean     = clean_acs(income_df,     "Income")
poverty_clean    = clean_acs(poverty_df,    "Poverty")
employment_clean = clean_acs(employment_df, "Employment")
housing_clean    = clean_acs(housing_df,    "Housing")

Cleaning ACS tables:
Age: (3221, 52) | FIPS dupes: 0
Race: (3221, 13) | FIPS dupes: 0
Education: (3221, 28) | FIPS dupes: 0
Income: (3221, 4) | FIPS dupes: 0
Poverty: (3221, 62) | FIPS dupes: 0
Employment: (3221, 10) | FIPS dupes: 0
Housing: (3221, 4) | FIPS dupes: 0


### (6) Prepare MIT 2020 election data

In [6]:
mit_2020_all = mit_df[mit_df["year"] == 2020].copy()
mit_2020_all = mit_2020_all[mit_2020_all["party"].isin(["DEMOCRAT", "REPUBLICAN"])].copy()
mit_2020_all = mit_2020_all[mit_2020_all["county_fips"].notna()].copy()

mit_2020_all["county_fips"] = (
    mit_2020_all["county_fips"]
    .astype(float).astype(int).astype(str).str.zfill(5)
)

# Counties that have a TOTAL row — use only those rows
has_total = set(mit_2020_all[mit_2020_all["mode"] == "TOTAL"]["county_fips"])

mit_total     = mit_2020_all[mit_2020_all["mode"] == "TOTAL"]
mit_non_total = mit_2020_all[~mit_2020_all["county_fips"].isin(has_total)]

mit_2020 = pd.concat([mit_total, mit_non_total], ignore_index=True)

print(f"Counties with TOTAL mode:    {len(has_total)}")
print(f"Counties without TOTAL mode: {mit_non_total['county_fips'].nunique()}")
print(f"Total rows after fix:        {len(mit_2020)}")
print(f"Total unique counties:       {mit_2020['county_fips'].nunique()}")

Counties with TOTAL mode:    2305
Counties without TOTAL mode: 849
Total rows after fix:        10064
Total unique counties:       3154


### (7) Pivot election data to one row per county

In [7]:
mit_pivot = mit_2020.pivot_table(
    index=["county_fips", "state", "county_name", "totalvotes"],
    columns="party",
    values="candidatevotes",
    aggfunc="sum"
).reset_index()

mit_pivot.columns.name = None

mit_pivot = mit_pivot.rename(columns={
    "DEMOCRAT":   "democrat_votes",
    "REPUBLICAN": "republican_votes"
})

mit_pivot["democrat_votes"]   = mit_pivot["democrat_votes"].fillna(0)
mit_pivot["republican_votes"] = mit_pivot["republican_votes"].fillna(0)

print(f"Shape: {mit_pivot.shape}")
print(f"Sample FIPS: {mit_pivot['county_fips'].head(5).tolist()}")
display(mit_pivot.head(3))

Shape: (3154, 6)
Sample FIPS: ['01001', '01003', '01005', '01007', '01009']


,county_fips,state,county_name,totalvotes,democrat_votes,republican_votes
0,01001,ALABAMA,AUTAUGA,27770,7503.0,19838.0
1,01003,ALABAMA,BALDWIN,109679,24578.0,83544.0
2,01005,ALABAMA,BARBOUR,10518,4816.0,5622.0


### (8) Create target variables

In [8]:
mit_pivot["party_winner"] = np.where(
    mit_pivot["democrat_votes"] > mit_pivot["republican_votes"], 1, 0
)

mit_pivot["dem_vote_share"] = (
    mit_pivot["democrat_votes"] / mit_pivot["totalvotes"]
)

print(mit_pivot["party_winner"].value_counts().rename({1:"Democrat", 0:"Republican"}))
print(f"\nAvg dem vote share: {mit_pivot['dem_vote_share'].mean():.3f}")
display(mit_pivot.head(3))

party_winner
Republican    2596
Democrat       558
Name: count, dtype: int64

Avg dem vote share: 0.334


,county_fips,state,county_name,totalvotes,democrat_votes,republican_votes,party_winner,dem_vote_share
0,01001,ALABAMA,AUTAUGA,27770,7503.0,19838.0,0,0.270184
1,01003,ALABAMA,BALDWIN,109679,24578.0,83544.0,0,0.224090
2,01005,ALABAMA,BARBOUR,10518,4816.0,5622.0,0,0.457882


### (9) Merge all ACS tables together

In [9]:
def slim(df):
    return df.drop(columns=["GEO_ID", "NAME"], errors="ignore")

acs_merged = age_clean.copy()

for df, label in [
    (race_clean,       "Race"),
    (education_clean,  "Education"),
    (income_clean,     "Income"),
    (poverty_clean,    "Poverty"),
    (employment_clean, "Employment"),
    (housing_clean,    "Housing"),
]:
    before = len(acs_merged)
    acs_merged = acs_merged.merge(slim(df), on="county_fips", how="inner")
    print(f"After merging {label}: {len(acs_merged)} counties (dropped {before - len(acs_merged)})")

print(f"\nFinal ACS merged shape: {acs_merged.shape}")

After merging Race: 3221 counties (dropped 0)
After merging Education: 3221 counties (dropped 0)
After merging Income: 3221 counties (dropped 0)
After merging Poverty: 3221 counties (dropped 0)
After merging Employment: 3221 counties (dropped 0)
After merging Housing: 3221 counties (dropped 0)

Final ACS merged shape: (3221, 155)


### (10) FIPS overlap check

In [10]:
acs_fips = set(acs_merged["county_fips"])
mit_fips = set(mit_pivot["county_fips"])

only_acs = acs_fips - mit_fips
only_mit = mit_fips - acs_fips
common   = acs_fips & mit_fips

print(f"ACS counties:        {len(acs_fips)}")
print(f"MIT counties:        {len(mit_fips)}")
print(f"Common (will merge): {len(common)}")
print(f"ACS only (dropped):  {len(only_acs)}")
print(f"MIT only (dropped):  {len(only_mit)}")

if only_acs:
    print(f"\nSample ACS-only FIPS: {sorted(only_acs)[:5]}")
if only_mit:
    print(f"\nSample MIT-only FIPS: {sorted(only_mit)[:5]}")

ACS counties:        3221
MIT counties:        3154
Common (will merge): 3115
ACS only (dropped):  106
MIT only (dropped):  39

Sample ACS-only FIPS: ['02050', '02060', '02063', '02066', '02068']

Sample MIT-only FIPS: ['02001', '02002', '02003', '02004', '02005']


In [11]:
# Check what modes exist in 2020 MIT data
#Temporary
mit_2020_all = mit_df[mit_df["year"] == 2020].copy()
print("All modes in 2020:")
print(mit_2020_all["mode"].value_counts())

# How many unique counties with TOTAL vs all modes
total_mode_counties = mit_2020_all[mit_2020_all["mode"] == "TOTAL"]["county_fips"].nunique()
all_mode_counties   = mit_2020_all["county_fips"].nunique()

print(f"\nUnique counties with TOTAL mode: {total_mode_counties}")
print(f"Unique counties across all modes: {all_mode_counties}")

All modes in 2020:
mode
TOTAL                   10059
ELECTION DAY             3737
ABSENTEE                 1995
PROVISIONAL              1832
ABSENTEE BY MAIL         1038
ONE STOP                  500
ADVANCED VOTING           477
PROV                      477
EARLY                     453
EARLY VOTE                450
FAILSAFE                  230
FAILSAFE PROVISIONAL      230
IN-PERSON ABSENTEE        230
MAIL                      145
2ND ABSENTEE              120
EARLY VOTING              120
Name: count, dtype: int64

Unique counties with TOTAL mode: 2305
Unique counties across all modes: 3154


### (11) Final merge — ACS + election targets

In [12]:
final_df = acs_merged.merge(mit_pivot, on="county_fips", how="inner")

# Remove any duplicate columns
final_df = final_df.loc[:, ~final_df.columns.duplicated()]

# Ensure FIPS stays as clean 5-digit string
final_df["county_fips"] = final_df["county_fips"].astype(str).str.zfill(5)

print(f"Final dataset shape: {final_df.shape}")
display(final_df.head(3))

Final dataset shape: (3115, 162)


,GEO_ID,NAME,county_fips,B01001_001E,B01001_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_007E,...,B23025_006E,B23025_007E,B25001_001E,state,county_name,totalvotes,democrat_votes,republican_votes,party_winner,dem_vote_share
0,0500000US01001,"Autauga County, Alabama",01001,55639,27052,1727,2156,1674,1208,553,...,709,18335,23697,ALABAMA,AUTAUGA,27770,7503.0,19838.0,0,0.270184
1,0500000US01003,"Baldwin County, Alabama",01003,218289,105889,6082,5197,8289,4305,2406,...,321,73736,116747,ALABAMA,BALDWIN,109679,24578.0,83544.0,0,0.224090
2,0500000US01005,"Barbour County, Alabama",01005,25026,13156,660,741,721,467,261,...,0,11051,12057,ALABAMA,BARBOUR,10518,4816.0,5622.0,0,0.457882


### (12) Missing values check

In [13]:
missing = final_df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing) == 0:
    print("No missing values in final dataset.")
else:
    print(f"{len(missing)} columns have missing values:")
    display(missing)

1 columns have missing values:


B19013_001E    1
dtype: int64

### (13) Save and verify

In [14]:
import os

os.makedirs("../data/processed", exist_ok=True)

final_df.to_csv("../data/processed/merged_county_dataset.csv", index=False)

print(f"Saved successfully!")
print(f"Shape: {final_df.shape}")
print(f"Republican-winning counties: {(final_df['party_winner']==0).sum()}")
print(f"Democrat-winning counties:   {(final_df['party_winner']==1).sum()}")

# Reload to verify
check = pd.read_csv(
    "../data/processed/merged_county_dataset.csv",
    dtype={"county_fips": str},
    low_memory=False
)
print(f"\nVerification reload shape: {check.shape}")

Saved successfully!
Shape: (3115, 162)
Republican-winning counties: 2576
Democrat-winning counties:   539

Verification reload shape: (3115, 162)
